# 03 – Giao thức đánh giá (Giai đoạn 3, bản sửa theo nhận xét)

**Checklist:** 3.1, 3.5, 3.6, 3.8 (sửa/mới) · 3.2, 3.3, 3.4, 3.7 (giữ nguyên)

Mọi phương pháp (baseline, bộ chọn thống kê, random search, ToT) đều dùng chung các hàm trong `src/evaluation.py`, để so sánh công bằng.

**Những gì thay đổi so với bản trước:**
1. **3 mức nhiệt độ.** Trước đây em dùng nhiệt độ thật của ngày D (ex-post), tức là đã dùng thông tin tương lai. Bây giờ tách ra:

| Mức | Nhiệt độ 24 giờ ngày D mà mô hình được thấy | Vai trò |
|---|---|---|
| L1 | không thấy; chỉ dùng nhiệt độ đến 23:00 D−1 | **bản chính** |
| L2 | nhiệt độ **dự báo** (notebook 04) | gần thực tế vận hành |
| L3 | nhiệt độ **thật** | oracle experiment, cận trên |

2. **Kiểm tra rò rỉ** phá hỏng cả nhiệt độ tương lai, không chỉ phụ tải.
3. **Ba cách chọn giai đoạn để chấm điểm** S1/S2/S3 (sẽ so sánh ở GĐ5).
4. **Origin đại diện** cho ToT.

In [1]:
import sys
sys.path.append("..")          # để import được thư mục src/ khi notebook nằm trong notebooks/

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

pd.set_option("display.width", 140)
pd.set_option("display.max_columns", 30)
plt.rcParams["figure.figsize"] = (11, 3.5)

from src.data import load_dataset, SELECTION_END
from src.evaluation import *
from src.models import build_model, BASELINES
from src.temperature import build_temp_forecast

df = load_dataset("system")
temp_fc, _, best_temp_method, _ = build_temp_forecast(df)   # cần cho mức L2, chi tiết ở notebook 04

## 3.1–3.2 Bài toán và cách chia thời gian

In [2]:
pd.DataFrame({
    "giai_doan": ["lịch sử (fit)", "chỉ dùng để chấm điểm", "validation", "test (dùng 1 lần)"],
    "tu": [DATA_START.date(), SCORE_START.date(), VAL_START.date(), TEST_START.date()],
    "den": [SELECTION_END.date(), (VAL_START - pd.Timedelta(days=1)).date(), VAL_END.date(), TEST_END.date()],
})

,giai_doan,tu,den
0,lịch sử (fit),2004-01-01,2006-06-30
1,chỉ dùng để chấm điểm,2005-07-01,2006-06-30
2,validation,2006-07-01,2007-06-30
3,test (dùng 1 lần),2007-07-01,2008-06-29


- **Origin:** 00:00 ngày D. **Horizon:** 24 giờ (00:00–23:00 ngày D).
- **Phụ tải** biết đến 23:00 ngày D−1, nên mọi đặc trưng phụ tải phải có độ trễ ≥ 24 giờ.
- **Biến lịch** (giờ, thứ, tháng, ngày lễ) biết trước nên dùng được cho cả 24 giờ.
- Các origin từ 01/07/2005 đến 30/06/2006 **không dùng để báo cáo**. Chúng chỉ cung cấp "quá khứ để chấm điểm" cho các bộ chọn ở đầu giai đoạn validation (S1 cần 365 ngày trước đó).

### Minh họa `information_set` ở 3 mức

Xem mô hình được thấy gì ở 3 giờ đầu của ngày D = 15/07/2006:

In [3]:
o = pd.Timestamp("2006-07-15")
show = {}
for lv in TEMP_LEVELS:
    info = information_set(df, o, lv, temp_fc, lookback_days=2)
    show[f"T ({lv})"] = info["T"]
show["load (mọi mức)"] = information_set(df, o, "L1", lookback_days=2)["load"]
pd.DataFrame(show).loc[o - pd.Timedelta(hours=2): o + pd.Timedelta(hours=2)]

,T (L1),T (L2),T (L3),load (mọi mức)
datetime,,,,
2006-07-14 22:00:00,75.181818,75.181818,75.181818,1880540.0
2006-07-14 23:00:00,74.363636,74.363636,74.363636,1731789.0
2006-07-15 00:00:00,NaN,73.467506,73.636364,NaN
2006-07-15 01:00:00,NaN,72.714542,73.272727,NaN
2006-07-15 02:00:00,NaN,72.017824,73.272727,NaN


Ở L1 nhiệt độ ngày D bị xóa (NaN), ở L2 là nhiệt độ dự báo, ở L3 là nhiệt độ thật. Phụ tải từ origin trở đi luôn bị xóa ở mọi mức. Vì dữ liệu tương lai bị xóa thật sự (chứ không chỉ "hứa không dùng"), nếu một mô hình ở L1 lỡ dùng T(t) thì dự báo sẽ ra NaN và lỗi ngay.

## 3.6 Kiểm tra rò rỉ cho mọi baseline ở cả 3 mức

Phá hỏng dữ liệu sau origin (nhân 1,5 rồi cộng 20) rồi dự báo lại:
- phá **phụ tải** tương lai → dự báo không được đổi ở mọi mức;
- phá **nhiệt độ** tương lai → ở L1, L2 không được đổi; ở L3 được phép đổi (vì L3 dùng nhiệt độ thật).

In [4]:
rows = []
o = pd.Timestamp("2006-08-09")
for name, cfg in BASELINES.items():
    for lv in TEMP_LEVELS:
        r = check_no_leakage(build_model(cfg, lv), df, o, lv, temp_fc)
        rows.append({"mo_hinh": name, "muc": lv,
                     "pha_load_doi": round(r["load"]["max_change"], 3),
                     "pha_T_doi": round(r["T"]["max_change"], 1),
                     "dat": r["load"]["ok"] and r["T"]["ok"]})
leak = pd.DataFrame(rows)
leak.to_csv("../results/tables/3_6_kiem_tra_ro_ri.csv", index=False)
print("Tất cả đạt:", leak["dat"].all())
leak.pivot(index="mo_hinh", columns="muc", values="pha_T_doi")

Tất cả đạt: True


muc,L1,L2,L3
mo_hinh,,,
lgbm,0.0,0.0,480014.3
mlr,0.0,0.0,5694443.8
mstl_ets,0.0,0.0,0.0
sarimax,0.0,0.0,915589.5
snaive168,0.0,0.0,0.0
snaive24,0.0,0.0,0.0


Bảng trên là mức thay đổi của dự báo khi phá nhiệt độ tương lai: chỉ cột L3 của các mô hình có dùng nhiệt độ là khác 0, đúng như thiết kế. Khi phá phụ tải tương lai, mọi ô đều bằng 0.

## 3.5 Ba cách chọn giai đoạn để chấm điểm

Tại mỗi điểm quyết định d (ngày 1 mỗi tháng), điểm của một cấu hình = MAPE trung bình của nó trên một tập origin **trước d**:

| Cách | Tập origin |
|---|---|
| S1 | toàn bộ 365 ngày trước d |
| S2 | cùng mùa (ngày 12–18 của tháng d trong 1–3 năm trước) + 7 ngày gần nhất |
| S3 | 21 ngày có **ngữ cảnh** giống ngày d nhất; nếu có ít hơn 10 ngày đủ gần thì quay về S2 |

Ngữ cảnh của một ngày (chỉ dùng thông tin đến 23:00 hôm trước): nhiệt độ TB và cao nhất hôm trước, nhiệt độ TB 7 ngày, mức tải 7 ngày so với 365 ngày, vị trí trong năm (sin/cos). Các đặc trưng được chuẩn hóa rồi so khoảng cách Euclid.

In [5]:
ctx = context_features(df)
rows = []
for d in decision_dates(VAL_START, TEST_END):
    s3, info = scoring_origins(d, "S3", ctx, return_info=True)
    rows.append({"d": d.date(), "S1": len(scoring_origins(d, "S1")),
                 "S2": len(scoring_origins(d, "S2")), "S3": len(s3), "S3_ghi_chu": info})
sizes = pd.DataFrame(rows)
print("Số tháng S3 phải fallback về S2:", (sizes["S3_ghi_chu"] != "S3").sum())
sizes.head(6)

Số tháng S3 phải fallback về S2: 0


,d,S1,S2,S3,S3_ghi_chu
0,2006-07-01,365,14,21,S3
1,2006-08-01,365,14,21,S3
2,2006-09-01,365,14,21,S3
3,2006-10-01,365,14,21,S3
4,2006-11-01,365,14,21,S3
5,2006-12-01,365,14,21,S3


In [6]:
# Ví dụ: S3 chọn những ngày nào cho d = 01/01/2007?
d = pd.Timestamp("2007-01-01")
s3 = scoring_origins(d, "S3", ctx)
print("Ngữ cảnh của d:"); print(ctx.loc[d].round(2).to_dict())
print("\nCác ngày S3 chọn:", [x.strftime("%Y-%m-%d") for x in s3])

Ngữ cảnh của d:
{'T_prev1': 45.69, 'Tmax_prev1': 51.73, 'T_prev7': 43.95, 'lvl_prev7': 1.05, 'doy_sin': 0.02, 'doy_cos': 1.0}

Các ngày S3 chọn: ['2005-12-29', '2005-12-30', '2005-12-31', '2006-01-01', '2006-01-02', '2006-01-03', '2006-01-04', '2006-01-05', '2006-01-06', '2006-01-09', '2006-01-11', '2006-01-18', '2006-12-17', '2006-12-22', '2006-12-25', '2006-12-26', '2006-12-27', '2006-12-28', '2006-12-29', '2006-12-30', '2006-12-31']


## 3.8 Origin đại diện cho ToT

Thầy yêu cầu chỉ chạy LLM trên một số thời điểm dự báo đại diện. Em lấy mẫu phân tầng theo:
- **mùa** (đông/xuân/hè/thu),
- **loại ngày** (thường / cuối tuần / lễ),
- **nhiệt độ** (cực trị = 10% nóng nhất hoặc lạnh nhất của mùa / bình thường),

mỗi tầng lấy 2 ngày. Danh sách được **cố định trước** khi chạy và lưu ra file. Số lượng origin sẽ chốt lại theo ý thầy.

In [7]:
rep_val = representative_origins(df, VAL_START, VAL_END)
rep_test = representative_origins(df, TEST_START, TEST_END)
rep_val.to_csv("../results/tables/3_8_origin_dai_dien_val.csv")
rep_test.to_csv("../results/tables/3_8_origin_dai_dien_test.csv")
print("validation:", len(rep_val), "origin | test:", len(rep_test), "origin")
rep_test.groupby(["season", "daytype", "temp"]).size().unstack("temp")

validation: 41 origin | test: 41 origin


temp              binh_thuong  cuc_tri
season daytype                        
dong   cuoi_tuan          2.0      2.0
       le                 2.0      2.0
       thuong             2.0      2.0
he     cuoi_tuan          2.0      2.0
       le                 1.0      NaN
       thuong             2.0      2.0
thu    cuoi_tuan          2.0      2.0
       le                 2.0      1.0
       thuong             2.0      2.0
xuan   cuoi_tuan          2.0      2.0
       le                 NaN      1.0
       thuong             2.0      2.0

## 3.7 Thử kiểm định Diebold–Mariano

So sánh seasonal naive lag 24 và lag 168 trên validation. Sai số được gộp theo ngày (1 con số / origin), phương sai dùng Newey–West, có hiệu chỉnh HLN.

In [8]:
from src.experiments import run_cached
fa = run_cached(df, BASELINES["snaive24"], "L1", start=VAL_START, end=VAL_END)
fb = run_cached(df, BASELINES["snaive168"], "L1", start=VAL_START, end=VAL_END)
print("MASE scale =", round(mase_scale(df)))
print(pd.DataFrame({"snaive24": metrics(fa, mase_scale(df)), "snaive168": metrics(fb, mase_scale(df))}).round(3))
dm_test(daily_loss(fa), daily_loss(fb))

  xong snaive_lag-24 [L1] trong 2s
  xong snaive_lag-168 [L1] trong 1s
MASE scale = 123893
        snaive24   snaive168
MAE   137340.080  231698.653
RMSE  189308.848  309610.750
MAPE       8.051      13.766
WAPE       8.141      13.734
MASE       1.109       1.870


{'mean_diff': np.float64(-5.714576721463085),
 'DM': np.float64(-7.341184790057643),
 'p_value': np.float64(1.39568179864627e-12),
 'n_days': 365}

`DM < 0` và p rất nhỏ: lag 24 tốt hơn lag 168 một cách có ý nghĩa thống kê. Kết quả hợp lý, vì với dự báo ngày tới, hôm qua giống hôm nay hơn tuần trước, trừ những ngày chuyển giữa cuối tuần và ngày thường.